# [Introduction | Online Z3 Guide](https://microsoft.github.io/z3guide/programming/Z3%20Python/Introduction)

## Install

In [8]:
import z3

print(z3.get_version_string())

5.3.3


In [9]:
from z3 import Int, solve

x = Int("x")
solve(x > 2, x < 5)

[x = 3]


## Getting Started

$$
\Gamma=
\left\{
\begin{aligned}
x&>2\\
y&<10\\
x+2y&=7
\end{aligned}
\right\}
$$

$$
\boxed{\exists x,y\in\mathbb Z:\bigwedge_{\varphi\in\Gamma}\varphi}
$$

In [10]:
from z3 import ArithRef, Int, solve

# 定数記号の宣言
x: ArithRef = Int("x")
y: ArithRef = Int("y")

solve(
    x > 2,
    y < 10,
    x + 2 * y == 7,
)

[y = 0, x = 7]


- = for assignment  
- The operators <, <=, >, >=, == and != for comparison.

In [11]:
from typing import cast

from z3 import BoolRef

constraint = cast(BoolRef, x + 2 * y == 7)

print(constraint)
print(type(constraint))

x + 2*y == 7
<class 'z3.z3.BoolRef'>


In [12]:
from z3 import simplify, And, Sum

print(simplify(Sum(x, y, 2 * x, 3)))
print(simplify(x < Sum(y, x, 2)))
print(
    simplify(
        And(
            cast(ArithRef, x + 1) >= 3,
            cast(ArithRef, Sum(x**2, x**2, y**2, 2)) >= 5,
        )
    )
)

3 + 3*x + y
Not(y <= -2)
And(x >= 2, 2*x**2 + y**2 >= 3)


### 数式表示モード

In [13]:
from z3 import set_option

set_option(html_mode=True)
print(cast(ArithRef, x**2 + y**2) >= 1)

set_option(html_mode=False)
print(cast(ArithRef, x**2 + y**2) >= 1)

x<sup>2</sup> + y<sup>2</sup> &ge; 1
x**2 + y**2 >= 1


### AST

In [14]:
x: ArithRef = cast(ArithRef, Int("x"))
y: ArithRef = cast(ArithRef, Int("y"))

expr: ArithRef = cast(ArithRef, Sum(x, y))
n: BoolRef = cast(BoolRef, expr >= 3)

print("num args: ", n.num_args())
print("children: ", n.children())
print("1st child:", n.arg(0))
print("2nd child:", n.arg(1))
print("operator: ", n.decl())
print("op name:  ", n.decl().name())

num args:  2
children:  [x + y, 3]
1st child: x + y
2nd child: 3
operator:  >=
op name:   >=


### nonlinear polynomial constraints

$$
\begin{cases}
x^2+y^2>3\\
x^3+y<5
\end{cases}
$$

In [ ]:
from typing import cast
from z3 import ArithRef, Real, solve

x: ArithRef = cast(ArithRef, Real("x"))
y: ArithRef = cast(ArithRef, Real("y"))

solve(
    x**2 + y**2 > 3,  # pyright: ignore[reportOperatorIssue]
    x**3 + y < 5,  # pyright: ignore[reportOperatorIssue]
)


[y = 2, x = 0]


代数的無理数(irrational algebraic numbers)

In [16]:
from z3 import set_option

solve(
    x**2 + y**2 == 3,  # pyright: ignore[reportOperatorIssue]
    x**3 == 2,
)

set_option(precision=30)
print("30桁表示:")
solve(
    x**2 + y**2 == 3,  # pyright: ignore[reportOperatorIssue]
    x**3 == 2,
)

[y = -1.1885280594?, x = 1.2599210498?]
30桁表示:
[y = -1.188528059421316533710369365015?,
 x = 1.259921049894873164767210607278?]


Rational numbers

In [18]:
from typing import cast
from z3 import ArithRef, Q, Real, RealVal

print(1 / 3)
print(RealVal(1) / 3)
print(Q(1, 3))

print()

x: ArithRef = cast(ArithRef, Real("x"))

print(x + 1 / 3)
print(x + Q(1, 3))
print(x + "1/3")
print(x + 0.25)

0.3333333333333333
1/3
1/3

x + 3333333333333333/10000000000000000
x + 1/3
x + 1/3
x + 1/4


In [20]:
from z3 import solve, set_option

set_option(rational_to_decimal=False)
solve(3 * x == 1)

set_option(rational_to_decimal=True)
solve(3 * x == 1)

set_option(precision=10)
solve(3 * x == 1)


[x = 1/3]
[x = 0.333333333333333333333333333333?]
[x = 0.3333333333?]


### Unsatisfiable

$$
\Gamma=\{x>4,\ x<0\}
$$

$$
\boxed{\Gamma\text{ is UNSAT}}
$$

In [21]:
from typing import cast
from z3 import ArithRef, Real, solve

x: ArithRef = cast(ArithRef, Real("x"))

solve(x > 4, x < 0)

no solution


## Boolean Logic

Z3 supports Boolean operators: And, Or, Not, Implies (implication), If (if-then-else). 

Bi-implications are represented using equality ==.

$$
\Gamma=
\left\{
\begin{aligned}
&p\rightarrow q\\
&r\leftrightarrow\neg q\\
&\neg p\lor r
\end{aligned}
\right\}
$$

In [ ]:
from typing import cast
from z3 import Bool, BoolRef, Implies, Not, Or, solve

p: BoolRef = cast(BoolRef, Bool("p"))
q: BoolRef = cast(BoolRef, Bool("q"))
r: BoolRef = cast(BoolRef, Bool("r"))

solve(
    Implies(p, q),
    r == Not(q),
    Or(Not(p), r),
)

no solution


And(p, q, True): $p\land q\land\top$  
And(p, q): $p\land q$  
False: $\bot$  

In [24]:
from typing import cast
from z3 import And, Bool, BoolRef, simplify

p: BoolRef = cast(BoolRef, Bool("p"))
q: BoolRef = cast(BoolRef, Bool("q"))

print(And(p, q, True))
print(simplify(And(p, q, True)))
print(simplify(And(p, False)))


And(p, q, True)
And(p, q)
False


a combination of polynomial and Boolean constraints

$$
\Gamma=
\left\{
\begin{aligned}
&x<5\lor x>10\\
&p\lor x^2=2\\
&\neg p
\end{aligned}
\right\}
$$

In [26]:
from typing import cast
from z3 import ArithRef, Bool, BoolRef, Not, Or, Real, solve

p: BoolRef = cast(BoolRef, Bool("p"))
x: ArithRef = cast(ArithRef, Real("x"))

solve(
    Or(x < 5, x > 10),
    Or(p, x**2 == 2),  # pyright: ignore[reportOperatorIssue]
    Not(p),
)

[x = -1.4142135623?, p = False]


## Solvers

add

$$\Gamma_0=\{x>10,\ y=x+2\}$$
SAT

---

push() → add(y < 11)

push

$$\Gamma_1=\Gamma_0\cup\{y<11\}$$
UNSAT

---

pop() → 追加制約を撤回

pop

$$\Gamma_0=\{x>10,\ y=x+2\}$$
SAT

In [28]:
from typing import cast
from z3 import ArithRef, Int, Solver

x: ArithRef = cast(ArithRef, Int("x"))
y: ArithRef = cast(ArithRef, Int("y"))

s: Solver = Solver()
print(s)

s.add(x > 10, y == x + 2)
print(s)
print("Solving constraints in the solver s ...")
print(s.check())

print()

print("Create a new scope...")
s.push()
s.add(y < 11)
print(s)
print("Solving updated set of constraints...")
print(s.check())

print()

print("Restoring state...")
s.pop()
print(s)
print("Solving restored set of constraints...")
print(s.check())


[]
[x > 10, y == x + 2]
Solving constraints in the solver s ...
sat

Create a new scope...
[x > 10, y == x + 2, y < 11]
Solving updated set of constraints...
unsat

Restoring state...
[x > 10, y == x + 2]
Solving restored set of constraints...
sat


Solverの unknown：制約集合の充足可能性を判定できなかった

In [30]:
from typing import cast
from z3 import ArithRef, Real, Solver, unknown

x: ArithRef = cast(ArithRef, Real("x"))
s: Solver = Solver()

s.add(2**x == 3)  # pyright: ignore[reportOperatorIssue]

result = s.check()
print(result)

unknown


Asserted Constraints と Statistics

In [33]:
from typing import cast
from z3 import ArithRef, Or, Real, Solver, Sum

x: ArithRef = cast(ArithRef, Real("x"))
y: ArithRef = cast(ArithRef, Real("y"))

s: Solver = Solver()
s.add(
    x > 1,
    y > 1,
    Or(
        cast(ArithRef, Sum(x, y)) > 3,
        cast(ArithRef, x - y) < 2,
    ),
)

print("asserted constraints...")
for c in s.assertions():
    print(c)

print()
print(s.check())
print()

print("statistics for the last check method...")
print(s.statistics())
print()

for k, v in s.statistics():
    print(k, ":", v)


asserted constraints...
x > 1
y > 1
Or(x + y > 3, x - y < 2)

sat

statistics for the last check method...
(:arith-bp-bound-candidates 1
 :arith-bp-rows-analyzed    4
 :arith-lower               1
 :arith-make-feasible       3
 :arith-max-columns         8
 :arith-max-rows            2
 :arith-upper               3
 :decisions                 2
 :final-checks              1
 :max-memory                19.81
 :memory                    17.56
 :mk-bool-var               4
 :mk-clause-binary          1
 :num-allocs                62980521
 :num-checks                1
 :rlimit-count              7440)

decisions : 2
final checks : 1
mk clause binary : 1
num checks : 1
mk bool var : 4
arith-lower : 1
arith-upper : 3
arith-make-feasible : 3
arith-bp-rows-analyzed : 4
arith-bp-bound-candidates : 1
arith-max-columns : 8
arith-max-rows : 2
num allocs : 62980521
rlimit count : 7440
max memory : 19.81
memory : 17.56


`s.model()`:	直前のSAT判定に対応するモデルを取得  
`m[x]`:	モデルにおける記号 $x$ の解釈を取得  
`m.decls()`:	モデル内で解釈が与えられた記号の宣言を列挙  

In [37]:
from typing import cast
from z3 import ArithRef, ModelRef, Real, Reals, Solver, sat

# x: ArithRef = cast(ArithRef, Real("x"))
# y: ArithRef = cast(ArithRef, Real("y"))
# z: ArithRef = cast(ArithRef, Real("z"))

x, y, z = Reals("x y z")

s: Solver = Solver()
s.add(
    x > 1,
    y > 1,
    x + y > 3,  # pyright: ignore[reportOperatorIssue]
    z - x < 10,  # pyright: ignore[reportOperatorIssue]
)

if s.check() == sat:
    m: ModelRef = s.model()
    print("x =", m[x])
    print()

    print("traversing model...")
    for d in m.decls():
        print(f"{d.name()} = {m[d]}")


x = 1.5

traversing model...
x = 1.5
z = 0
y = 2


In [36]:
from typing import cast
from z3 import ArithRef, Int, Real, Reals, Ints, ToReal, Sum

x: ArithRef = cast(ArithRef, Real("x"))
y: ArithRef = cast(ArithRef, Int("y"))

a, b, c = Reals("a b c")
s, r = Ints("s r")

print(Sum(x, y, 1, a, s))
print(Sum(ToReal(y), c))


x + ToReal(y) + 1 + a + ToReal(s)
ToReal(y) + c


## Arithmetic